# Binance API - Khám phá sàn chứng khoán crypto Binance API

Notebook này giúp bạn khám phá nhanh api `Binance API`:
- Kiểm tra kết nối API qua biến môi trường
- Kiểm tra dữ liệu real-time qua websocket
- Kiểm tra những sàn giao dịch hỗ trợ
- Lấy dữ liệu nến từ quá khứ
- Lấy tin tức chung của sàn chứng khoán

## 1. Cấu hình

In [1]:
import sys
import os
import asyncio
import pandas as pd
from datetime import datetime
from IPython.display import clear_output, display

# Thêm thư mục gốc vào path để import src
sys.path.append(os.path.abspath('..'))

from src.config import Config
from src.streamer import stream_market_data

## 2. Giao dịch - Cập nhật giá cuối cùng

Finhub có hỗ trợ cho bản free Websocket để lấy giá liên tục từ chứng khoán thị trường Hoa Kỳ

In [2]:
import requests

def get_active_usdt_symbols(limit=1000):
    """
    Tự động kéo danh sách các mã giao dịch Spot đang hoạt động trên Binance.
    """
    url = "https://api.binance.com/api/v3/exchangeInfo"
    
    try:
        response = requests.get(url)
        response.raise_for_status() # Bắt lỗi nếu gọi API thất bại
        data = response.json()
        
        symbols = []
        for s in data["symbols"]:
            # Điều kiện lọc: Cặp giao dịch với USDT và Đang mở giao dịch
            if s["quoteAsset"] == "USDT" and s["status"] == "TRADING":
                symbols.append(s["symbol"].lower())
                
        print(f"Tổng số mã USDT đang hoạt động trên Binance: {len(symbols)}")
        
        # Trả về số lượng giới hạn theo yêu cầu (tối đa 1024 cho 1 WS)
        return symbols[:limit]
        
    except Exception as e:
        print(f"Lỗi khi lấy danh sách mã: {e}")
        return []

# Sử dụng:
symbols = get_active_usdt_symbols(1000)
print(symbols)

Tổng số mã USDT đang hoạt động trên Binance: 439
['btcusdt', 'ethusdt', 'bnbusdt', 'neousdt', 'ltcusdt', 'qtumusdt', 'adausdt', 'xrpusdt', 'tusdusdt', 'iotausdt', 'xlmusdt', 'ontusdt', 'trxusdt', 'etcusdt', 'icxusdt', 'vetusdt', 'usdcusdt', 'linkusdt', 'ongusdt', 'hotusdt', 'zilusdt', 'zrxusdt', 'fetusdt', 'batusdt', 'zecusdt', 'iostusdt', 'celrusdt', 'dashusdt', 'thetausdt', 'enjusdt', 'atomusdt', 'tfuelusdt', 'oneusdt', 'algousdt', 'dogeusdt', 'duskusdt', 'ankrusdt', 'winusdt', 'cosusdt', 'mtlusdt', 'dentusdt', 'wanusdt', 'funusdt', 'cvcusdt', 'chzusdt', 'bandusdt', 'xtzusdt', 'rvnusdt', 'hbarusdt', 'stxusdt', 'kavausdt', 'arpausdt', 'iotxusdt', 'rlcusdt', 'bchusdt', 'fttusdt', 'eurusdt', 'ognusdt', 'lskusdt', 'bntusdt', 'mblusdt', 'cotiusdt', 'solusdt', 'ctsiusdt', 'hiveusdt', 'chrusdt', 'ardrusdt', 'mdtusdt', 'kncusdt', 'compusdt', 'scusdt', 'zenusdt', 'snxusdt', 'vthousdt', 'dgbusdt', 'dcrusdt', 'storjusdt', 'manausdt', 'yfiusdt', 'jstusdt', 'crvusdt', 'sandusdt', 'nmrusdt', 'dotu

Demo một vài mã

In [ ]:
import asyncio
from collections import defaultdict
from datetime import datetime
from IPython.display import clear_output

# Danh sách symbols demo
symbol_demo = symbols[0:10]

# 1. Khởi tạo State kép
state = {
    "trade": {
        "counter": defaultdict(int),
        "max_per_symbol": defaultdict(int),
        "total_max": 0
    },
    "depth": {
        "counter": defaultdict(int),
        "max_per_symbol": defaultdict(int),
        "total_max": 0
    }
}

# 2. Consumer Task: Phân loại và xử lý dữ liệu từ stream
async def process_all_data(symbols):
    """Hút dữ liệu trộn lẫn và phân loại vào state tương ứng"""
    async for payload in stream_market_data(symbols, stream_type="all", speed="1000ms"):
        stream_name = payload["stream"]
        data = payload["data"]
        symbol = data["s"].upper()
        
        if "@trade" in stream_name:
            state["trade"]["counter"][symbol] += 1
        elif "@depth" in stream_name:
            # Depth update trả về danh sách các mức giá thay đổi (bids 'b' và asks 'a')
            update_count = len(data["b"]) + len(data["a"])
            state["depth"]["counter"][symbol] += update_count

# 3. UI Task: Hiển thị 2 bảng song song (trên-dưới)
async def display_dual_dashboard(symbols):
    symbols_upper = [s.upper() for s in symbols]
    
    while True:
        await asyncio.sleep(1)
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # --- Xử lý dữ liệu Trade ---
        t_counts = state["trade"]["counter"].copy()
        t_total = sum(t_counts.values())
        state["trade"]["counter"].clear()
        
        if t_total > state["trade"]["total_max"]:
            state["trade"]["total_max"] = t_total
            
        # --- Xử lý dữ liệu Depth ---
        d_counts = state["depth"]["counter"].copy()
        d_total = sum(d_counts.values())
        state["depth"]["counter"].clear()
        
        if d_total > state["depth"]["total_max"]:
            state["depth"]["total_max"] = d_total

        # --- Giao diện ---
        clear_output(wait=True)
        print(f"🚀 DUAL MARKET MONITOR | {current_time}")
        
        # BẢNG 1: TRADE STATS
        print("\n" + "="*25 + " 💰 TRADE STATS " + "="*25)
        print(f"{'Symbol':<12} | {'Count/s':<10} | {'Max/s':<10}")
        print("-" * 60)
        for s in symbols_upper:
            if t_counts[s] > state["trade"]["max_per_symbol"][s]:
                state["trade"]["max_per_symbol"][s] = t_counts[s]
            print(f"{s:<12} | {t_counts[s]:<10} | {state['trade']['max_per_symbol'][s]:<10}")
        print("-" * 60)
        print(f"{'TOTAL TRADE':<12} | {t_total:<10} | {state['trade']['total_max']:<10}")

        # BẢNG 2: ORDER BOOK (DEPTH) STATS
        print("\n" + "="*25 + " 📖 DEPTH STATS " + "="*25)
        print(f"{'Symbol':<12} | {'Updates/s':<10} | {'Max/s':<10}")
        print("-" * 60)
        for s in symbols_upper:
            if d_counts[s] > state["depth"]["max_per_symbol"][s]:
                state["depth"]["max_per_symbol"][s] = d_counts[s]
            print(f"{s:<12} | {d_counts[s]:<10} | {state['depth']['max_per_symbol'][s]:<10}")
        print("-" * 60)
        print(f"{'TOTAL DEPTH':<12} | {d_total:<10} | {state['depth']['total_max']:<10}")
        print("=" * 66)

# 4. Main
async def run_dual_engine():    
    consumer = asyncio.create_task(process_all_data(symbol_demo))
    ui = asyncio.create_task(display_dual_dashboard(symbol_demo))
    await asyncio.gather(consumer, ui)

# Chạy hệ thống
task = asyncio.create_task(run_dual_engine())

🚀 DUAL MARKET MONITOR | 2026-04-21 14:02:12

========================= 💰 TRADE STATS =========================
Symbol       | Count/s    | Max/s     
------------------------------------------------------------
BTCUSDT      | 3          | 424       
ETHUSDT      | 8          | 294       
BNBUSDT      | 0          | 87        
NEOUSDT      | 0          | 0         
LTCUSDT      | 0          | 7         
QTUMUSDT     | 0          | 0         
ADAUSDT      | 16         | 19        
XRPUSDT      | 0          | 122       
TUSDUSDT     | 0          | 0         
IOTAUSDT     | 0          | 1         
------------------------------------------------------------
TOTAL TRADE  | 27         | 839       

========================= 📖 DEPTH STATS =========================
Symbol       | Updates/s  | Max/s     
------------------------------------------------------------
BTCUSDT      | 0          | 1707      
ETHUSDT      | 0          | 739       
BNBUSDT      | 31         | 190       
NEOUSDT      | 

2026-04-21 14:02:22,752 | INFO     | Nhận lệnh tắt. Đóng WebSocket an toàn...


Nếu muốn dừng, bạn chạy lệnh dưới đây

In [4]:
task.cancel()

True

Tính toán tất cả mã crypto cho trade

In [ ]:
import asyncio
import time
from datetime import datetime
from IPython.display import clear_output

# 1. State tối giản cho Stress Test
state = {
    "current_count": 0,
    "max_total": 0,
    "last_max_event_time": 0, # Thời gian sự kiện lớn nhất từng ghi nhận
    "backward_count": 0,      # Đếm số lần tin nhắn đến muộn (giật lùi)
    "latencies": []           # Lưu trữ độ trễ mạng trong 1 giây
}

# 2. Consumer Task: Tập trung đếm và đo lường thời gian
async def process_data_stress_test(symbols):
    async for msg in stream_market_data(symbols, stream_type="trade"):
        state["current_count"] += 1
        
        # 'E' là Event time (thời gian sự kiện) từ Binance tính bằng mili-giây
        event_time = msg["data"]["E"] 
        
        # Lấy thời gian hiện tại của máy tính tính bằng mili-giây
        local_time = int(time.time() * 1000) 
        
        # Tính độ trễ mạng (Lưu ý: Nếu đồng hồ máy tính bạn chạy sai, số này có thể âm)
        latency = local_time - event_time
        state["latencies"].append(latency)

        # Kiểm tra hiện tượng "Giật lùi" (Out-of-order)
        if event_time < state["last_max_event_time"]:
            state["backward_count"] += 1
        else:
            state["last_max_event_time"] = event_time

# 3. UI Task: Giao diện tổng quan (Overview Dashboard)
async def display_stress_dashboard():
    while True:
        await asyncio.sleep(1)
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # Lấy snapshot dữ liệu
        total = state["current_count"]
        latencies = state["latencies"].copy()
        backward = state["backward_count"]
        
        # Reset state cho giây tiếp theo
        state["current_count"] = 0
        state["latencies"].clear()
        state["backward_count"] = 0
        
        # Cập nhật Max
        if total > state["max_total"]:
            state["max_total"] = total
            
        # Tính toán thống kê độ trễ (tránh chia cho 0 nếu không có data)
        avg_latency = sum(latencies) / len(latencies) if latencies else 0
        max_latency = max(latencies) if latencies else 0
        
        # Vẽ bảng
        clear_output(wait=True)
        print("=" * 60)
        print(f"🚀 STRESS TEST DASHBOARD | Cập nhật lúc: {current_time}")
        print("-" * 60)
        print(f"Tổng số tin nhắn/giây : {total:<8} | Max kỷ lục: {state['max_total']}")
        print("-" * 60)
        print(f"Độ trễ trung bình     : {avg_latency:.2f} ms")
        print(f"Độ trễ cao nhất       : {max_latency} ms")
        print(f"Lỗi Out-of-order      : {backward} tin nhắn (bị giật lùi)")
        print("=" * 60)

# 4. Main Task
async def run_stress_engine(symbols):
    # Khởi tạo 2 luồng
    consumer = asyncio.create_task(process_data_stress_test(symbols))
    ui = asyncio.create_task(display_stress_dashboard())
    
    await asyncio.gather(consumer, ui)

task = asyncio.create_task(run_stress_engine(symbols))

🚀 STRESS TEST DASHBOARD | Cập nhật lúc: 2026-04-21 14:02:31
------------------------------------------------------------
Tổng số tin nhắn/giây : 89       | Max kỷ lục: 747
------------------------------------------------------------
Độ trễ trung bình     : -527.97 ms
Độ trễ cao nhất       : -501 ms
Lỗi Out-of-order      : 0 tin nhắn (bị giật lùi)


2026-04-21 14:02:38,315 | INFO     | Nhận lệnh tắt. Đóng WebSocket an toàn...


In [6]:
task.cancel()

True

Depth - order book: dữ liệu nặng và nhiều hơn so với trade

In [ ]:
# 1. Khởi tạo State chuyên dụng cho Depth Stress Test
state = {
    "msg_count": 0,           # Đếm số lượng gói tin JSON bay về
    "level_updates": 0,       # Đếm tổng số mức giá Bids + Asks thay đổi
    "max_msg_per_sec": 0,     # Kỷ lục số tin nhắn/giây
    "max_level_updates": 0,   # Kỷ lục số mức giá cập nhật/giây
    "last_max_event_time": 0, # Thời gian sự kiện lớn nhất từng ghi nhận (để check giật lùi)
    "backward_count": 0,      # Số lượng tin nhắn đến muộn
    "latencies": []           # Mảng lưu độ trễ mạng trong 1 giây
}

# 2. Consumer Task: Bóc tách Payload của Depth
async def process_depth_stress(symbols, stream_type="depth", speed="100ms"):
    """Hút dữ liệu từ luồng Depth và tính toán độ trễ"""
    # Gọi hàm stream_market_data (đã định nghĩa ở cell trước) với speed 100ms
    async for payload in stream_market_data(symbols, stream_type=stream_type, speed=speed):
        data = payload["data"]
        
        # 1. Cập nhật số gói tin
        state["msg_count"] += 1
        
        # 2. Tính số lượng mức giá thay đổi (bids 'b' và asks 'a')
        # Binance trả về mảng rỗng [] nếu không có thay đổi ở phía đó
        bids_len = len(data.get("b", []))
        asks_len = len(data.get("a", []))
        state["level_updates"] += (bids_len + asks_len)
        
        # 3. Tính toán độ trễ (Latency)
        event_time = data["E"] # Thời gian Binance đóng gói tin nhắn
        local_time = int(time.time() * 1000) # Thời gian máy tính của bạn nhận được
        
        latency = local_time - event_time
        state["latencies"].append(latency)
        
        # 4. Kiểm tra Out-of-order (Giật lùi thời gian)
        if event_time < state["last_max_event_time"]:
            state["backward_count"] += 1
        else:
            state["last_max_event_time"] = event_time

# 3. UI Task: Vẽ Dashboard tối giản cho Stress Test
async def display_depth_stress_dashboard():
    while True:
        await asyncio.sleep(1)
        
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # Lấy snapshot dữ liệu của 1 giây vừa qua
        msg_total = state["msg_count"]
        level_total = state["level_updates"]
        latencies = state["latencies"].copy()
        backward = state["backward_count"]
        
        # Reset state ngay lập tức cho giây tiếp theo
        state["msg_count"] = 0
        state["level_updates"] = 0
        state["latencies"].clear()
        state["backward_count"] = 0
        
        # Cập nhật kỷ lục Max
        if msg_total > state["max_msg_per_sec"]:
            state["max_msg_per_sec"] = msg_total

        if level_total > state["max_level_updates"]:
            state["max_level_updates"] = level_total
            
        # Tính toán thống kê Latency
        avg_latency = sum(latencies) / len(latencies) if latencies else 0
        max_latency = max(latencies) if latencies else 0
        
        # Xóa màn hình và in Dashboard
        clear_output(wait=True)
        print("=" * 65)
        print(f"📖 DEPTH STRESS TEST DASHBOARD | Cập nhật lúc: {current_time}")
        print("-" * 65)
        print(f"Số gói tin mạng/giây    : {msg_total:<8} | Max kỷ lục: {state['max_msg_per_sec']}")
        print(f"Số mức giá cập nhật/giây: {level_total:<8} | Max kỷ lục: {state['max_level_updates']}")
        print("(Tổng Bids + Asks)")
        print("-" * 65)
        print(f"Độ trễ trung bình       : {avg_latency:.2f} ms")
        print(f"Độ trễ cao nhất         : {max_latency} ms")
        print(f"Lỗi Out-of-order        : {backward} tin nhắn (bị giật lùi)")
        print("=" * 65)

# 4. Main Task: Điều phối
async def run_depth_stress_engine():
    consumer = asyncio.create_task(process_depth_stress(symbols, stream_type="depth", speed="100ms"))
    ui = asyncio.create_task(display_depth_stress_dashboard())
    await asyncio.gather(consumer, ui)

# Kích hoạt hệ thống
task = asyncio.create_task(run_depth_stress_engine())

📖 DEPTH STRESS TEST DASHBOARD | Cập nhật lúc: 2026-04-21 14:03:29
-----------------------------------------------------------------
Số gói tin mạng/giây    : 1032     | Max kỷ lục: 2487
Số mức giá cập nhật/giây: 3938     | Max kỷ lục: 11310
(Tổng Bids + Asks)
-----------------------------------------------------------------
Độ trễ trung bình       : 5845.73 ms
Độ trễ cao nhất         : 6313 ms
Lỗi Out-of-order        : 0 tin nhắn (bị giật lùi)


2026-04-21 14:03:40,501 | INFO     | Nhận lệnh tắt. Đóng WebSocket an toàn...


In [8]:
task.cancel()

True

Tiếp tục là test cả trade và depth, tổng cộng là 878 luồng

In [10]:
import asyncio
import json
import time
import logging
import websockets
from datetime import datetime
from IPython.display import display, HTML
from websockets.exceptions import ConnectionClosedError

# Tắt log rác
logging.getLogger('websockets').setLevel(logging.ERROR)

BASE_URL = "wss://stream.binance.com:9443/stream"

# ==========================================
# 1. BỘ NHỚ ĐỆM (STATE) TỔNG HỢP
# ==========================================
state = {
    "trade": {
        "current_count": 0,
        "max_count": 0
    },
    "depth": {
        "current_msg": 0,
        "current_levels": 0,
        "max_msg": 0,
        "max_levels": 0
    },
    "latencies": [] # Dùng chung để đo độ trễ mạng tổng thể
}

# ==========================================
# 2. HÀM KẾT NỐI API (SUPER STREAM)
# ==========================================
async def stream_market_data(symbols: list, display_handle):
    if not symbols: return
    
    # Gộp cả 2 loại stream vào chung 1 đường truyền
    streams_to_subscribe = []
    streams_to_subscribe.extend([f"{s.lower()}@trade" for s in symbols])
    streams_to_subscribe.extend([f"{s.lower()}@depth@100ms" for s in symbols])

    while True:
        try:
            async with websockets.connect(BASE_URL) as ws:
                chunk_size = 150
                for i in range(0, len(streams_to_subscribe), chunk_size):
                    chunk = streams_to_subscribe[i:i + chunk_size]
                    await ws.send(json.dumps({"method": "SUBSCRIBE", "params": chunk, "id": i + 1}))
                    await asyncio.sleep(0.5) 
                
                while True:
                    msg = await ws.recv()
                    payload = json.loads(msg)
                    if "stream" in payload and "data" in payload:
                        yield payload
                        
        except Exception as e:
            display_handle.update(HTML(f"<div style='color:#f44336; font-family:monospace;'>⚠️ Lỗi mạng: {e}. Đang kết nối lại...</div>"))
            await asyncio.sleep(5)

# ==========================================
# 3. TASK BÓC TÁCH DỮ LIỆU (CONSUMER)
# ==========================================
async def process_market_data(symbols, display_handle):
    async for payload in stream_market_data(symbols, display_handle):
        stream_name = payload["stream"]
        data = payload["data"]
        
        # --- Tính độ trễ chung ---
        event_time = data["E"] 
        local_time = int(time.time() * 1000) 
        state["latencies"].append(local_time - event_time)
        
        # --- Phân loại dữ liệu ---
        if "@trade" in stream_name:
            state["trade"]["current_count"] += 1
            
        elif "@depth" in stream_name:
            state["depth"]["current_msg"] += 1
            bids_len = len(data.get("b", []))
            asks_len = len(data.get("a", []))
            state["depth"]["current_levels"] += (bids_len + asks_len)

# ==========================================
# 4. TASK GIAO DIỆN (UI DASHBOARD KÉP)
# ==========================================
async def display_unified_dashboard(display_handle):
    while True:
        await asyncio.sleep(1)
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # Lấy snapshot dữ liệu Trade
        t_count = state["trade"]["current_count"]
        state["trade"]["current_count"] = 0
        if t_count > state["trade"]["max_count"]:
            state["trade"]["max_count"] = t_count
            
        # Lấy snapshot dữ liệu Depth
        d_msg = state["depth"]["current_msg"]
        d_levels = state["depth"]["current_levels"]
        state["depth"]["current_msg"] = 0
        state["depth"]["current_levels"] = 0
        
        if d_msg > state["depth"]["max_msg"]:
            state["depth"]["max_msg"] = d_msg
        if d_levels > state["depth"]["max_levels"]:
            state["depth"]["max_levels"] = d_levels
            
        # Tính toán Latency
        lats = state["latencies"].copy()
        state["latencies"].clear()
        avg_lat = sum(lats) / len(lats) if lats else 0
        max_lat = max(lats) if lats else 0
        
        # Render HTML
        html_content = f"""
        <div style="background-color: #1e1e1e; color: #d4d4d4; padding: 20px; font-family: Consolas, monospace; border-radius: 8px; width: fit-content; min-width: 450px;">
            <h3 style="color: #4CAF50; margin-top: 0; margin-bottom: 5px; text-align: center;">⚡ UNIFIED MARKET DATA ENGINE</h3>
            <div style="color: #858585; margin-bottom: 15px; font-size: 0.9em; text-align: center;">Live at: {current_time}</div>
            
            <div style="border-bottom: 1px solid #444; margin-bottom: 10px; padding-bottom: 5px;">
                <b style="color: #569cd6;">📊 LƯU LƯỢNG TRADE (Khớp lệnh thực tế)</b>
            </div>
            <table style="width: 100%; text-align: left; margin-bottom: 15px;">
                <tr><td style="width: 60%;">Số giao dịch/giây:</td><td><b style="color: #ce9178;">{t_count}</b></td></tr>
                <tr><td style="color: #858585;">Max kỷ lục (Spike):</td><td style="color: #858585;">{state['trade']['max_count']} trades/s</td></tr>
            </table>

            <div style="border-bottom: 1px solid #444; margin-bottom: 10px; padding-bottom: 5px;">
                <b style="color: #C586C0;">📖 LƯU LƯỢNG DEPTH (Sổ lệnh)</b>
            </div>
            <table style="width: 100%; text-align: left; margin-bottom: 15px;">
                <tr><td style="width: 60%;">Gói tin mạng/giây:</td><td><b style="color: #b5cea8;">{d_msg}</b></td></tr>
                <tr><td style="color: #858585;">Max gói tin (Spike):</td><td style="color: #858585;">{state['depth']['max_msg']} msgs/s</td></tr>
                <tr><td style="padding-top: 5px;"><b>Mức giá thay đổi/giây:</b></td><td style="padding-top: 5px;"><b style="color: #dcdcaa; font-size: 1.1em;">{d_levels}</b></td></tr>
                <tr><td style="color: #858585;">Max thay đổi (Spike):</td><td style="color: #858585;">{state['depth']['max_levels']} updates/s</td></tr>
            </table>
            
            <div style="border-bottom: 1px solid #444; margin-bottom: 10px; padding-bottom: 5px;">
                <b style="color: #4EC9B0;">🌐 TÌNH TRẠNG MẠNG</b>
            </div>
            <table style="width: 100%; text-align: left;">
                <tr><td style="width: 60%;">Độ trễ trung bình:</td><td>{avg_lat:.2f} ms</td></tr>
                <tr><td>Độ trễ Max:</td><td><span style="color: {'#f44336' if max_lat > 500 else '#d4d4d4'};">{max_lat} ms</span></td></tr>
            </table>
        </div>
        """
        display_handle.update(HTML(html_content))

# ==========================================
# 5. HÀM ĐIỀU PHỐI VÀ KÍCH HOẠT
# ==========================================
async def run_unified_engine(display_handle):
    consumer = asyncio.create_task(process_market_data(symbols, display_handle))
    ui = asyncio.create_task(display_unified_dashboard(display_handle))
    
    done, pending = await asyncio.wait([consumer, ui], return_when=asyncio.FIRST_EXCEPTION)
    for t in done:
        if t.exception():
            display_handle.update(HTML(f"<div style='color:red; font-weight:bold;'>🚨 CRASH: {t.exception()}</div>"))

# Quản lý luồng Jupyter
if 'task' in globals() and not task.done():
    task.cancel()
    print("🛑 Đã dừng cỗ máy cũ. Đang khởi động luồng mới...")

# Tạo màn hình chờ
dashboard_display = display(HTML("<div style='font-family: monospace; padding: 20px;'>⏳ Khởi động Unified Engine...</div>"), display_id=True)

# Kích hoạt hệ thống
task = asyncio.create_task(run_unified_engine(dashboard_display))

🛑 Đã dừng cỗ máy cũ. Đang khởi động luồng mới...


Số giao dịch/giây:,16
Max kỷ lục (Spike):,339 trades/s
Gói tin mạng/giây:,90
Max gói tin (Spike):,1124 msgs/s
Mức giá thay đổi/giây:,294
Max thay đổi (Spike):,4362 updates/s
Độ trễ trung bình:,24395.93 ms
Độ trễ Max:,24449 ms


In [12]:
task.cancel()

False